# Stage 2/3/4 Atlas-Free CNN Brain-Text Pipeline

Downstream runner for the controlled Stage 2/3/4 experiments. Stage 1A and Stage 1B autoencoder training and checkpoint-level held-out evaluations are upstream completed artifacts from notebooks 6 and 7. This notebook loads the completed evaluation outputs, validates the empirically selected AE checkpoints, and writes downstream manifests before any Stage 2/3/4 work begins.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match the working Colab setup used by Notebook 2:
# code repo in /content/neurovlm_gnn, Drive used for data and run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

from atlas_free_cnn.pipeline_outputs import (
    create_stage2_stage3_stage4_run_dir,
    git_info,
    write_json,
    write_status_report,
    write_readme_what_to_look_at,
    write_table,
    flatten_stage3_metrics,
    ae_source_metric_columns,
)
from stage1_selection_integration import IntegrationConfig, integrate_completed_stage1_selection

## Config Switches

Stage 1 is not runnable from this notebook. The required inputs are the four explicit empirically selected autoencoder checkpoint paths. Completed Stage 1 checkpoint-evaluation folders are optional provenance inputs only; they are not needed to decide which checkpoint to load.

In [ ]:
def split_dir_has_jsonl(path: Path) -> bool:
    return all((path / name).exists() for name in ["train.jsonl", "val.jsonl", "test.jsonl"])


HF_DATASET_REPO = os.environ.get("NEUROVLM_ATLAS_FREE_HF_REPO", "neurovlm/atlas_free_cnn_dataset")
LOCAL_UNIFIED_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild"
LOCAL_SPLIT_DIR = LOCAL_UNIFIED_CACHE_DIR / "splits"
LOCAL_PACK_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn_rebuild"


def hf_download_first_available(filenames, local_dir: Path) -> Path:
    from huggingface_hub import hf_hub_download
    local_dir.mkdir(parents=True, exist_ok=True)
    errors = []
    for filename in filenames:
        try:
            path = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                filename=filename,
                local_dir=str(local_dir),
                local_dir_use_symlinks=False,
            )
            return Path(path)
        except Exception as exc:
            errors.append(f"{filename}: {exc}")
    raise FileNotFoundError("Could not download any candidate from HF:\n" + "\n".join(errors))


def ensure_hf_unified_splits() -> Path:
    print(f"Downloading atlas-free CNN split JSONLs from Hugging Face: {HF_DATASET_REPO}")
    LOCAL_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "val", "test"]:
        downloaded = hf_download_first_available(
            [f"splits/{split}.jsonl", f"unified_jsonl_rebuild/splits/{split}.jsonl", f"{split}.jsonl"],
            LOCAL_UNIFIED_CACHE_DIR,
        )
        target = LOCAL_SPLIT_DIR / f"{split}.jsonl"
        if downloaded.resolve() != target.resolve():
            shutil.copy2(downloaded, target)
    for name in ["train_map_ids.json", "val_map_ids.json", "test_map_ids.json"]:
        try:
            downloaded = hf_download_first_available(
                [f"splits/{name}", f"unified_jsonl_rebuild/splits/{name}", name],
                LOCAL_UNIFIED_CACHE_DIR,
            )
            target = LOCAL_SPLIT_DIR / name
            if downloaded.resolve() != target.resolve():
                shutil.copy2(downloaded, target)
        except Exception as exc:
            print(f"Optional split sidecar not downloaded ({name}): {exc}")
    try:
        downloaded_volume = hf_download_first_available(
            ["atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn/atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn_rebuild/atlas_free_cnn_volumes.pt"],
            LOCAL_PACK_DIR,
        )
        target_volume = LOCAL_PACK_DIR / "atlas_free_cnn_volumes.pt"
        if downloaded_volume.resolve() != target_volume.resolve():
            try:
                if target_volume.exists() or target_volume.is_symlink():
                    target_volume.unlink()
                os.symlink(downloaded_volume, target_volume)
            except Exception:
                shutil.copy2(downloaded_volume, target_volume)
        print("Volume tensor available at:", target_volume)
    except Exception as exc:
        print("WARNING: split JSONLs downloaded, but volume tensor was not prepared:", exc)
        print("Training will fail unless tensor_path values inside JSONL resolve to an accessible tensor file.")
    return LOCAL_SPLIT_DIR


def discover_unified_split_dir() -> Path:
    override = os.environ.get("NEUROVLM_UNIFIED_SPLIT_DIR")
    candidates = []
    if override:
        candidates.append(Path(override))
    candidates.extend([
        REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "cache/unified_jsonl/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/unified_jsonl/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/cache/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_atlas_free_cnn/cache/unified_jsonl/splits",
        DRIVE_ROOT / "data_ale_3dcnn/unified_jsonl_rebuild/splits",
        DRIVE_ROOT / "data_ale_3dcnn/unified_jsonl/splits",
    ])
    for candidate in candidates:
        if split_dir_has_jsonl(candidate):
            return candidate
    try:
        hf_split_dir = ensure_hf_unified_splits()
        if split_dir_has_jsonl(hf_split_dir):
            return hf_split_dir
    except Exception as exc:
        hf_error = exc
    else:
        hf_error = None
    checked = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find unified dataset split JSONL files locally, and Hugging Face fallback did not produce them. Expected train.jsonl, val.jsonl, and test.jsonl in one of:\n"
        f"{checked}\n\n"
        f"HF dataset repo tried: {HF_DATASET_REPO}\n"
        f"HF fallback error: {hf_error}\n\n"
        "If your splits are elsewhere, set os.environ['NEUROVLM_UNIFIED_SPLIT_DIR'] to that splits directory before running this cell."
    )

UNIFIED_SPLIT_DIR = discover_unified_split_dir()
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
print("Unified split dir:", UNIFIED_SPLIT_DIR)
print("Train JSONL:", TRAIN_JSONL)

RUN_MODE = "downstream_only"
DATA_MODE = "mixed"
RERUN_STAGE1_CHECKPOINT_EVALUATION = False

# Required: explicit selected checkpoint paths. Set NEUROVLM_AE_CHECKPOINT_ROOT to the parent
# that contains mixed_baseline_raw_mse/, pubmed/, nilearn/, and neurovault/, or set each
# NEUROVLM_*_AE_CKPT variable directly.
AE_CHECKPOINT_ROOT_VALUE = os.environ.get("NEUROVLM_AE_CHECKPOINT_ROOT", "").strip()
AE_CHECKPOINT_ROOT = Path(AE_CHECKPOINT_ROOT_VALUE).expanduser() if AE_CHECKPOINT_ROOT_VALUE else DRIVE_ROOT / "EDIT_ME_AE_CHECKPOINT_ROOT"

CONFIGURED_SELECTED_AE_CHECKPOINTS = {
    "mixed_stage1a": {
        "path": os.environ.get(
            "NEUROVLM_MIXED_STAGE1A_AE_CKPT",
            str(AE_CHECKPOINT_ROOT / "mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt"),
        ),
        "stage": "stage1a",
        "training_domain": "mixed",
        "checkpoint_name": "best_top1_dice.pt",
        "selection_reason": "held_out_multi_source_rank_1",
        "evaluation_status": "completed",
    },
    "mixed_to_pubmed_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_PUBMED_STAGE1B_AE_CKPT",
            str(AE_CHECKPOINT_ROOT / "pubmed/checkpoints/best_top1_dice.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "pubmed",
        "checkpoint_name": "best_top1_dice.pt",
        "selection_reason": "held_out_domain_rank_1",
        "evaluation_status": "completed",
    },
    "mixed_to_nilearn_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_NILEARN_STAGE1B_AE_CKPT",
            str(AE_CHECKPOINT_ROOT / "nilearn/checkpoints/best_val_loss.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "nilearn",
        "checkpoint_name": "best_val_loss.pt",
        "selection_reason": "held_out_top5_dice_rank_1",
        "evaluation_status": "completed",
    },
    "mixed_to_neurovault_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_NEUROVAULT_STAGE1B_AE_CKPT",
            str(AE_CHECKPOINT_ROOT / "neurovault/checkpoints/best_top5_dice.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "neurovault",
        "checkpoint_name": "best_top5_dice.pt",
        "selection_reason": "held_out_top5_dice_rank_1",
        "evaluation_status": "completed",
    },
}

# Optional provenance-only notebook-7 evaluation output folders. Leave blank if unavailable.
STAGE1A_EVALUATION_DIR_VALUE = os.environ.get("NEUROVLM_STAGE1A_EVALUATION_DIR", "").strip()
STAGE1B_EVALUATION_DIR_VALUE = os.environ.get("NEUROVLM_STAGE1B_EVALUATION_DIR", "").strip()
STAGE1A_EVALUATION_DIR = Path(STAGE1A_EVALUATION_DIR_VALUE).expanduser() if STAGE1A_EVALUATION_DIR_VALUE else None
STAGE1B_EVALUATION_DIR = Path(STAGE1B_EVALUATION_DIR_VALUE).expanduser() if STAGE1B_EVALUATION_DIR_VALUE else None

RUN_STAGE1_SELECTION_INTEGRATION = True
RUN_STAGE3_CONTRASTIVE = True
RUN_STAGE4_TEXT_TO_BRAIN = False
RUN_STAGE5_GENERATION_EVAL = False

NUM_WORKERS = int(os.environ.get("NEUROVLM_NUM_WORKERS", "4" if IN_COLAB else "0"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", str(NUM_WORKERS)))
PREFETCH_FACTOR = int(os.environ.get("NEUROVLM_PREFETCH_FACTOR", "4"))
METRICS_DEVICE = os.environ.get("NEUROVLM_METRICS_DEVICE", "cuda")

BASE_OUTPUT_DIR = DRIVE_ROOT / "runs_stage2_stage3_stage4" if "DRIVE_ROOT" in globals() else Path("runs_stage2_stage3_stage4")
STAGE1_SELECTION_INTEGRATION_OUTPUT_ROOT = DRIVE_ROOT / "runs_stage1_selection_integration" if "DRIVE_ROOT" in globals() else Path("runs_stage1_selection_integration")

In [ ]:
paths = create_stage2_stage3_stage4_run_dir(BASE_OUTPUT_DIR)
RUN_DIR = Path(paths["run_dir"])
print(f"Run directory: {RUN_DIR}")

metadata_dir = Path(paths["metadata"])
write_json(metadata_dir / "run_config.json", {
    "RUN_MODE": RUN_MODE,
    "DATA_MODE": DATA_MODE,
    "AE_CHECKPOINT_ROOT": str(AE_CHECKPOINT_ROOT),
    "CONFIGURED_SELECTED_AE_CHECKPOINTS": CONFIGURED_SELECTED_AE_CHECKPOINTS,
    "STAGE1A_EVALUATION_DIR": str(STAGE1A_EVALUATION_DIR or ""),
    "STAGE1B_EVALUATION_DIR": str(STAGE1B_EVALUATION_DIR or ""),
    "RERUN_STAGE1_CHECKPOINT_EVALUATION": RERUN_STAGE1_CHECKPOINT_EVALUATION,
    "RUN_STAGE1_SELECTION_INTEGRATION": RUN_STAGE1_SELECTION_INTEGRATION,
    "RUN_STAGE3_CONTRASTIVE": RUN_STAGE3_CONTRASTIVE,
    "RUN_STAGE4_TEXT_TO_BRAIN": RUN_STAGE4_TEXT_TO_BRAIN,
    "RUN_STAGE5_GENERATION_EVAL": RUN_STAGE5_GENERATION_EVAL,
})
write_json(metadata_dir / "git_info.json", git_info(REPO_DIR))
(metadata_dir / "environment.txt").write_text(sys.version)

## Stage 1 Selection Integration: Validate Explicit Checkpoint Paths

In [ ]:
SELECTED_AE_CHECKPOINTS = {}
STAGE2_STAGE3_STAGE4_INPUT_MANIFEST = None
STAGE1_SELECTION_INTEGRATION_DIR = None
stage1_selection_status = "not_requested"

if RUN_STAGE1_SELECTION_INTEGRATION:
    integration_result = integrate_completed_stage1_selection(
        IntegrationConfig(
            output_root=STAGE1_SELECTION_INTEGRATION_OUTPUT_ROOT,
            selected_checkpoints=CONFIGURED_SELECTED_AE_CHECKPOINTS,
            stage1a_evaluation_dir=STAGE1A_EVALUATION_DIR,
            stage1b_evaluation_dir=STAGE1B_EVALUATION_DIR,
            rerun_stage1_checkpoint_evaluation=RERUN_STAGE1_CHECKPOINT_EVALUATION,
        )
    )
    stage1_selection_status = integration_result["status"]
    STAGE1_SELECTION_INTEGRATION_DIR = Path(integration_result["output_dir"])
    STAGE2_STAGE3_STAGE4_INPUT_MANIFEST = STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/stage2_stage3_stage4_input_manifest.json"
    downstream_manifest = json.loads(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST.read_text())
    SELECTED_AE_CHECKPOINTS = downstream_manifest["selected_ae_checkpoints"]
    write_json(metadata_dir / "stage1_selection_integration_result.json", integration_result)
    write_json(metadata_dir / "stage2_stage3_stage4_input_manifest_pointer.json", {"path": str(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST)})
    if stage1_selection_status not in {"completed", "completed_with_warnings"}:
        raise RuntimeError(f"Stage 1 selection integration failed: {stage1_selection_status}")
    print("Stage 1 checkpoint validation:", stage1_selection_status)
    print("Integration output:", STAGE1_SELECTION_INTEGRATION_DIR)
    print("Downstream manifest:", STAGE2_STAGE3_STAGE4_INPUT_MANIFEST)
else:
    print("Stage 1 selection integration not requested")

## Preserved Stage 1 Evaluation Tables

In [ ]:
if STAGE1_SELECTION_INTEGRATION_DIR:
    for path in [
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/mixed_stage1a_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/pubmed_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/nilearn_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/neurovault_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_ae_checkpoints_for_stage2_stage3_stage4.json",
        STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_checkpoint_validation.json",
        STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/six_run_ae_assignment.csv",
    ]:
        print(path, "exists=", path.exists())
else:
    print("No Stage 1 selection integration output available")

## Stage 2/3: Six Controlled Encoder Initialization Runs

In [ ]:
if RUN_STAGE3_CONTRASTIVE:
    if not STAGE2_STAGE3_STAGE4_INPUT_MANIFEST or not Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).exists():
        raise RuntimeError("Stage 2/3 requires the validated Stage 1 selected-checkpoint manifest")
    downstream_manifest = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text())
    domain_dirs = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
    specialized_dirs = {
        "pubmed": "specialized_mixed_to_pubmed",
        "nilearn": "specialized_mixed_to_nilearn",
        "neurovault": "specialized_mixed_to_neurovault",
    }
    for run in downstream_manifest["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        stage3_dir = RUN_DIR / domain_dirs[domain] / branch / "stage3"
        ckpt_dir = stage3_dir / "checkpoints"
        ae_entry = downstream_manifest["selected_ae_checkpoints"][run["ae_registry_key"]]
        checkpoint_name = Path(ae_entry["checkpoint_name"]).stem
        cmd = [
            sys.executable, "experiments/3dcnn/train_ale_cnn.py",
            "--mode", "atlas_free",
            "--model", "ale_3dcnn",
            "--encoder-init", "autoencoder_pretrained",
            "--ae-ckpt-path", str(ae_entry["path"]),
            "--ae-init-variant", str(run["ae_registry_key"]),
            "--ae-checkpoint-selection", checkpoint_name,
            "--text-proj-init", "pretrained_infonce",
            "--run-dir", str(stage3_dir),
            "--checkpoint-dir", str(ckpt_dir),
        ]
        print("Launching", run["run"])
        print(" ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print("Stage 3 not requested")

# Stage 4: Run All Six Controlled Variants

Stage 4 is required for all six Stage 2/3 variants. Do not run Stage 4 only for the Stage 3 winners.

The purpose of this experiment is to determine whether domain-specific Stage 1B autoencoder fine-tuning improves:

1. Stage 3 bidirectional brain-to-text and text-to-brain retrieval.
2. Stage 4 text-to-brain generation.

All Stage 4 comparisons must be made within the same domain using identical train, validation, and test splits.

## Empirically selected AE checkpoints

Use the checkpoint choices established by the completed held-out Stage 1 checkpoint evaluations:

* General mixed Stage 1A:
  `mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt`

* Mixed→PubMed Stage 1B:
  `pubmed/checkpoints/best_top1_dice.pt`

* Mixed→Nilearn Stage 1B:
  `nilearn/checkpoints/best_val_loss.pt`

* Mixed→NeuroVault Stage 1B:
  `neurovault/checkpoints/best_top5_dice.pt`

Do not replace these with checkpoints selected only from their filenames or validation labels.

The checkpoint registry must use explicit paths and record that these were selected from held-out checkpoint comparisons.

Example:

```python
AE_CHECKPOINT_REGISTRY = {
    "mixed_stage1a": {
        "path": ".../mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt",
        "stage": "stage1a",
        "domain": "mixed",
        "selection_metric": "held_out_multi_source_rank_1",
    },
    "mixed_to_pubmed_stage1b": {
        "path": ".../pubmed/checkpoints/best_top1_dice.pt",
        "stage": "stage1b",
        "domain": "pubmed",
        "selection_metric": "held_out_domain_rank_1",
    },
    "mixed_to_nilearn_stage1b": {
        "path": ".../nilearn/checkpoints/best_val_loss.pt",
        "stage": "stage1b",
        "domain": "nilearn",
        "selection_metric": "held_out_top5_dice_rank_1",
    },
    "mixed_to_neurovault_stage1b": {
        "path": ".../neurovault/checkpoints/best_top5_dice.pt",
        "stage": "stage1b",
        "domain": "neurovault",
        "selection_metric": "held_out_top5_dice_rank_1",
    },
}
```

Do not infer checkpoints using:

* directory order;
* modification time;
* whichever model ran last;
* a generic `best_cnn_autoencoder.pt` alias;
* checkpoint names alone.

# Required Six-Run Stage 4 Matrix

## PubMed

### 1. `mixed_stage1a_on_pubmed_stage4`

* AE checkpoint:
  mixed Stage 1A `best_top1_dice.pt`
* AE decoder:
  decoder loaded from that exact mixed Stage 1A checkpoint
* Stage 3 source:
  `mixed_stage1a_on_pubmed`
* Stage 4 train data:
  PubMed train split
* Stage 4 validation data:
  PubMed validation split
* Final generation evaluation:
  PubMed test split

### 2. `mixed_to_pubmed_stage1b_on_pubmed_stage4`

* AE checkpoint:
  mixed→PubMed Stage 1B `best_top1_dice.pt`
* AE decoder:
  decoder loaded from that exact PubMed Stage 1B checkpoint
* Stage 3 source:
  `mixed_to_pubmed_stage1b_on_pubmed`
* Stage 4 train data:
  exact same PubMed train split as the baseline
* Stage 4 validation data:
  exact same PubMed validation split as the baseline
* Final generation evaluation:
  exact same PubMed test split as the baseline

## Nilearn

### 3. `mixed_stage1a_on_nilearn_stage4`

* AE checkpoint:
  mixed Stage 1A `best_top1_dice.pt`
* AE decoder:
  decoder loaded from that exact mixed Stage 1A checkpoint
* Stage 3 source:
  `mixed_stage1a_on_nilearn`
* Stage 4 train data:
  Nilearn train split
* Stage 4 validation data:
  Nilearn validation split
* Final generation evaluation:
  Nilearn test split

### 4. `mixed_to_nilearn_stage1b_on_nilearn_stage4`

* AE checkpoint:
  mixed→Nilearn Stage 1B `best_val_loss.pt`
* AE decoder:
  decoder loaded from that exact Nilearn Stage 1B checkpoint
* Stage 3 source:
  `mixed_to_nilearn_stage1b_on_nilearn`
* Stage 4 train data:
  exact same Nilearn train split as the baseline
* Stage 4 validation data:
  exact same Nilearn validation split as the baseline
* Final generation evaluation:
  exact same Nilearn test split as the baseline

## NeuroVault

### 5. `mixed_stage1a_on_neurovault_stage4`

* AE checkpoint:
  mixed Stage 1A `best_top1_dice.pt`
* AE decoder:
  decoder loaded from that exact mixed Stage 1A checkpoint
* Stage 3 source:
  `mixed_stage1a_on_neurovault`
* Stage 4 train data:
  NeuroVault train split
* Stage 4 validation data:
  NeuroVault validation split
* Final generation evaluation:
  NeuroVault test split

### 6. `mixed_to_neurovault_stage1b_on_neurovault_stage4`

* AE checkpoint:
  mixed→NeuroVault Stage 1B `best_top5_dice.pt`
* AE decoder:
  decoder loaded from that exact NeuroVault Stage 1B checkpoint
* Stage 3 source:
  `mixed_to_neurovault_stage1b_on_neurovault`
* Stage 4 train data:
  exact same NeuroVault train split as the baseline
* Stage 4 validation data:
  exact same NeuroVault validation split as the baseline
* Final generation evaluation:
  exact same NeuroVault test split as the baseline

# Required Component Matching

Each Stage 4 run must load components only from its corresponding upstream experiment.

For every Stage 4 variant, explicitly load:

1. The Stage 3 text-side representation and projection from the matching Stage 3 run.
2. The AE decoder from the exact Stage 1A or Stage 1B checkpoint that initialized that Stage 3 variant.
3. The correct domain-specific train, validation, and test splits.
4. A new Stage 4 text-to-latent projection head belonging only to that run.

The valid component paths are:

```text
Mixed Stage 1A best_top1_dice AE
→ mixed AE decoder
→ corresponding domain-specific mixed-baseline Stage 3 checkpoint
→ corresponding Stage 4 projection

Mixed→PubMed Stage 1B best_top1_dice AE
→ PubMed-fine-tuned decoder
→ PubMed-specialized Stage 3 checkpoint
→ PubMed-specialized Stage 4 projection

Mixed→Nilearn Stage 1B best_val_loss AE
→ Nilearn-fine-tuned decoder
→ Nilearn-specialized Stage 3 checkpoint
→ Nilearn-specialized Stage 4 projection

Mixed→NeuroVault Stage 1B best_top5_dice AE
→ NeuroVault-fine-tuned decoder
→ NeuroVault-specialized Stage 3 checkpoint
→ NeuroVault-specialized Stage 4 projection
```

Do not mix components across variants.

Invalid examples include:

* PubMed Stage 3 text projection with the Nilearn decoder.
* A specialized Stage 3 checkpoint with the general mixed decoder.
* A mixed Stage 3 checkpoint with a Stage 1B specialized decoder.
* NeuroVault Stage 4 evaluated on the Nilearn test set.
* One Stage 4 projection head reused across variants.
* `best_top5_dice.pt` used for PubMed even though the empirical selected checkpoint is `best_top1_dice.pt`.
* `best_top5_dice.pt` used for Nilearn even though the empirical selected checkpoint is `best_val_loss.pt`.
* `best_spatial_corr.pt` used as the mixed baseline even though the completed held-out comparison selected `best_top1_dice.pt`.

Fail before training if provenance does not match.

# Stage 4 Component Provenance Validation

Before each Stage 4 run, generate:

```text
stage4_component_provenance.json
```

It must contain:

* Stage 4 run name
* domain
* baseline or specialized status
* AE checkpoint path
* AE checkpoint filename
* AE stage: Stage 1A or Stage 1B
* AE training domain
* AE held-out selection reason
* AE checkpoint epoch
* AE encoder checksum
* AE decoder checksum
* Stage 3 run name
* Stage 3 checkpoint path
* Stage 3 checkpoint epoch
* Stage 3 checkpoint selection metric
* Stage 3 text projection checksum
* SPECTER model/version
* Stage 4 train split fingerprint
* Stage 4 validation split fingerprint
* Stage 4 test split fingerprint
* decoder trainable status
* Stage 3 text projection trainable status
* Stage 4 text-to-latent projection trainable status

The notebook must verify:

* Stage 3 domain equals Stage 4 domain.
* Stage 3 AE initialization path equals the AE checkpoint that supplied the Stage 4 decoder.
* The decoder checksum matches the decoder from the registered AE checkpoint.
* Baseline and specialized runs within a domain use identical split fingerprints.
* No held-out test example appears in Stage 4 train or validation data.
* Test data is evaluation-only.

# Stage 4 Architecture

Preserve the established text-to-brain generation procedure unless the codebase contains a previously validated configuration that differs.

The intended path is:

```text
domain text
→ SPECTER embedding
→ matching Stage 3 text projection
→ trainable Stage 4 text-to-AE-latent projection
→ matching frozen AE decoder
→ generated brain map
```

Codex must inspect the current implementation and verify exactly which Stage 3 text representation is used.

For the primary six-run experiment:

* AE decoder: frozen
* Stage 4 text-to-AE-latent projection: trainable
* Stage 3 CNN encoder: not used for generation updates
* Stage 3 text projection: frozen by default
* SPECTER base model or precomputed embeddings: unchanged
* AE latent dimension: 384
* decoder output shape: `[batch_size, 1, 36, 45, 38]`

If the previously validated Stage 4 recipe trained the Stage 3 text projection, preserve that behavior identically across all six runs and log it clearly.

Do not use different trainability settings for baseline and specialized variants.

Save:

```text
stage4_trainable_parameter_report.json
```

with trainable and frozen parameter counts for:

* SPECTER/base text model
* Stage 3 text projection
* Stage 4 text-to-latent projection
* AE decoder
* any adapter or normalization layer

# Stage 4 Dimensional Preflight

Before full training, verify:

* raw text/SPECTER input dimension
* Stage 3 projected text dimension
* Stage 4 input dimension
* Stage 4 output dimension
* AE latent dimension
* decoder input dimension
* decoder output shape

Run a preflight forward pass:

```text
text batch
→ valid SPECTER/text embedding
→ Stage 3 projected text representation
→ Stage 4 latent tensor [batch_size, 384]
→ decoder output [batch_size, 1, 36, 45, 38]
```

Fail immediately if any dimension is incompatible.

# Fairness Controls

Within each domain, baseline and specialized Stage 4 runs must use identical:

* ordered train map IDs
* ordered validation map IDs
* ordered test map IDs
* selected text IDs
* primary-text policy
* SPECTER embeddings
* batching policy
* optimizer
* learning rate
* scheduler
* epoch budget
* early-stopping policy
* loss function
* random seed where practical
* checkpoint-selection policy
* evaluation implementation

The only intended differences within a domain are:

1. Mixed Stage 1A versus domain-specialized Stage 1B AE/decoder.
2. The corresponding Stage 3-aligned text representation.

Required fingerprint equality:

```text
PubMed:
mixed Stage 1A Stage 4 splits
==
mixed→PubMed Stage 1B Stage 4 splits

Nilearn:
mixed Stage 1A Stage 4 splits
==
mixed→Nilearn Stage 1B Stage 4 splits

NeuroVault:
mixed Stage 1A Stage 4 splits
==
mixed→NeuroVault Stage 1B Stage 4 splits
```

Fail before training if a within-domain pair uses different examples.

# Stage 4 Loss and Checkpointing

Inspect the existing Stage 4 code and expose the complete configuration. Do not rely on hidden defaults.

Save for every Stage 4 run:

* `best_val_loss.pt`
* `best_val_spatial_corr.pt`
* `best_val_top5_dice.pt`
* `best_val_foreground_mse.pt`
* `last.pt`

Do not select the final generation checkpoint using validation MSE alone.

Use a documented generation-checkpoint policy emphasizing:

1. validation spatial correlation;
2. validation top-5 Dice or overlap;
3. foreground reconstruction quality;
4. validation MSE as a secondary measure.

Log separately:

* total loss
* raw MSE
* foreground MSE
* latent-space loss, if used
* spatial correlation
* top-1 Dice
* top-5 Dice
* top-10 Dice
* learning rate
* gradient norm
* epoch runtime
* GPU memory use

# Final Generation Evaluation

After Stage 4 training, evaluate every variant on its matching held-out test split.

The test split must never be used for:

* gradient updates;
* early stopping;
* checkpoint selection;
* hyperparameter selection.

For every test example, save:

* map ID
* text ID
* source/domain
* input text
* target map reference
* generated map reference
* MSE
* MAE
* foreground MSE
* spatial correlation
* top-1 Dice
* top-5 Dice
* top-10 Dice
* top-1 overlap
* top-5 overlap
* top-10 overlap
* target nonzero fraction
* predicted nonzero fraction
* predicted mean
* predicted maximum
* voxel AUROC when meaningful

Save generated maps or a compact reproducible prediction tensor plus a manifest mapping tensor indices to map and text IDs.

# Stage 4 Paired Comparisons

Create:

```text
pubmed_stage4_baseline_vs_specialized.csv
nilearn_stage4_baseline_vs_specialized.csv
neurovault_stage4_baseline_vs_specialized.csv
```

For each metric, include:

* baseline value
* specialized value
* absolute difference
* relative percentage difference where meaningful
* winner
* paired bootstrap confidence interval if practical
* whether the confidence interval includes zero

Compare at minimum:

* spatial correlation
* top-1 Dice
* top-5 Dice
* top-10 Dice
* foreground MSE
* raw MSE
* MAE

Also create:

```text
all_domain_stage4_comparison.csv
```

Do not rank PubMed, Nilearn, and NeuroVault raw values as though the domains have identical difficulty. Primary conclusions must come from within-domain paired comparisons.

# Combined Stage 1–4 Comparison

Create:

```text
ae_retrieval_generation_comparison.csv
```

Each row must represent one of the six variants and include:

* domain
* baseline or specialized
* AE variant
* AE checkpoint path
* AE checkpoint selection reason
* AE held-out spatial correlation
* AE held-out top-5 Dice
* Stage 3 run
* Stage 3 checkpoint
* Stage 3 mean recall@1
* Stage 3 mean recall@5
* Stage 3 mean recall@10
* Stage 3 mean recall@50
* Stage 3 mean MRR
* Stage 3 mean median rank
* Stage 3 mean AUC
* Stage 4 run
* Stage 4 checkpoint
* generation spatial correlation
* generation top-1 Dice
* generation top-5 Dice
* generation top-10 Dice
* generation foreground MSE
* generation raw MSE
* warnings
* status

Use this table to examine:

* whether Stage 1B reconstruction improvements correspond to Stage 3 retrieval improvements;
* whether Stage 3 retrieval improvements correspond to Stage 4 generation improvements;
* whether specialization helps retrieval but hurts generation, or vice versa;
* whether the effect differs by domain.

Do not imply causality from simple correlations.

# Output Organization

Use:

```text
runs_stage2_stage3_stage4/
  stage2_stage3_stage4_<timestamp>/
    00_run_metadata/
      master_config.json
      ae_checkpoint_registry.json
      split_fingerprints.json
      run_status.json

    01_pubmed/
      baseline_mixed_stage1a/
        stage2/
        stage3/
        stage4/
          00_config/
          01_provenance/
          02_training/
            checkpoints/
            metrics/
            plots/
          03_generation_evaluation/
            per_example/
            summaries/
            generated_maps/
            plots/

      specialized_mixed_to_pubmed/
        stage2/
        stage3/
        stage4/
          same structure

      comparison/
        pubmed_stage3_baseline_vs_specialized.csv
        pubmed_stage4_baseline_vs_specialized.csv
        pubmed_full_pipeline_comparison.csv

    02_nilearn/
      baseline_mixed_stage1a/
      specialized_mixed_to_nilearn/
      comparison/

    03_neurovault/
      baseline_mixed_stage1a/
      specialized_mixed_to_neurovault/
      comparison/

    04_all_domain_comparison/
      all_domain_stage3_comparison.csv
      all_domain_stage4_comparison.csv
      ae_retrieval_generation_comparison.csv
      domain_winners.json
      run_status.csv
      README_WHAT_TO_LOOK_AT.md
```

Do not overwrite Stage 2 or Stage 3 outputs when Stage 4 begins.

# Completion Detection

A Stage 4 run is complete only if all required outputs exist and are valid:

* Stage 4 config
* provenance report
* trainable-parameter report
* valid Stage 4 checkpoint
* training history
* validation history
* held-out generation metrics
* nonzero number of predictions
* per-example metrics
* generated-map manifest

Use statuses:

* `not_requested`
* `pending`
* `running`
* `completed`
* `completed_with_warnings`
* `failed`
* `incomplete_outputs`

Do not mark a run complete merely because a cell executed or an in-memory variable exists.

The overall pipeline cannot be marked complete unless all six Stage 4 variants have valid outputs.

# Final README Questions

The final README must answer:

1. Did PubMed-specific AE fine-tuning improve PubMed Stage 3 retrieval?
2. Did PubMed-specific AE fine-tuning improve PubMed Stage 4 generation?
3. Did Nilearn-specific AE fine-tuning improve Nilearn Stage 3 retrieval?
4. Did Nilearn-specific AE fine-tuning improve Nilearn Stage 4 concept-to-map generation?
5. Did NeuroVault-specific AE fine-tuning improve NeuroVault Stage 3 retrieval?
6. Did NeuroVault-specific AE fine-tuning improve NeuroVault Stage 4 description-to-statistical-map generation?
7. Were effects consistent across reconstruction, retrieval, and generation?
8. Which pipeline is recommended for each domain?
9. Were all within-domain comparisons based on identical splits?
10. Were any runs incomplete or affected by warnings?

# Acceptance Criteria

This implementation is complete only when:

* all six Stage 4 variants run;
* the correct empirically selected AE checkpoint is used for every variant;
* every Stage 4 run uses its matching Stage 3 checkpoint;
* every Stage 4 run uses the decoder from the exact matching AE checkpoint;
* no component is mixed across domains or baseline/specialized paths;
* baseline and specialized runs use identical domain-specific splits;
* all decoders use the same frozen/trainable policy;
* all Stage 4 runs use the same optimization recipe;
* test sets remain evaluation-only;
* generation predictions and metrics are saved for all six runs;
* three within-domain Stage 4 comparison files are produced;
* the combined Stage 1/3/4 comparison table is produced;
* run completion is based on actual saved outputs.

In [ ]:
if RUN_STAGE4_TEXT_TO_BRAIN:
    print("Configure and launch atlas_free_cnn.training.train_text_to_brain here; outputs should go under", Path(paths["stage4"]))
else:
    print("Stage 4 not requested")

if RUN_STAGE5_GENERATION_EVAL:
    print("Configure and launch generation evaluation here; outputs should go under", Path(paths["stage5"]))
else:
    print("Stage 5 not requested")

## Final Status and Comparison Files

In [ ]:
stage_status = write_status_report(RUN_DIR, {
    "stage1_selection_integration": stage1_selection_status,
    "stage3": RUN_STAGE3_CONTRASTIVE,
    "stage4": RUN_STAGE4_TEXT_TO_BRAIN,
    "stage5": RUN_STAGE5_GENERATION_EVAL,
})

if STAGE1_SELECTION_INTEGRATION_DIR:
    assignment_csv = STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/six_run_ae_assignment.csv"
    final_assignment = Path(paths["final"]) / "six_run_ae_assignment.csv"
    if assignment_csv.exists():
        shutil.copy2(assignment_csv, final_assignment)
    write_table(Path(paths["final"]) / "final_summary_table.csv", [{"stage": s["stage"], "status": s["status"]} for s in stage_status])
    write_json(Path(paths["final"]) / "stage1_selection_integration_pointer.json", {
        "integration_dir": str(STAGE1_SELECTION_INTEGRATION_DIR),
        "downstream_manifest": str(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST),
    })
else:
    write_table(Path(paths["final"]) / "final_summary_table.csv", [{"stage": s["stage"], "status": s["status"]} for s in stage_status])

for s in stage_status:
    print(f"{s['stage']}: {s['status']}")
print("Final comparison:", Path(paths["final"]))